In [1]:
%cd ..

/Users/philipphager/Documents/GitHub/clix


In [2]:
import pandas as pd
import altair as alt

from notebooks import theme
from pathlib import Path

In [3]:
result_dir = Path("clax-results/4-baidu-ultr-features/")
df = pd.concat([pd.read_csv(f) for f in list(result_dir.glob("*/test_*.csv"))], ignore_index=True)

mixture_result_dir = Path("clax-results/5-mixture-model/")
mixture_df = pd.concat([pd.read_csv(f) for f in list(mixture_result_dir.glob("*/test_*.csv"))], ignore_index=True)
mixture_df["model"] = "Mixture"

df = pd.concat([mixture_df, df])
df.head()

,model,test_loss,test_ll,test_ppl,test_cond_ppl,train_time_s,test_dcg@10,test_dcg@5,test_dcg@3,test_dcg@1,test_mrr@10
0,Mixture,1.707669,-0.186389,1.227624,1.189055,1709.947374,NaN,NaN,NaN,NaN,NaN
1,Mixture,NaN,NaN,NaN,NaN,NaN,7.774142,5.062751,3.667769,1.715247,0.624711
2,Mixture,1.710595,-0.187062,1.228790,1.190072,1632.485996,NaN,NaN,NaN,NaN,NaN
3,Mixture,NaN,NaN,NaN,NaN,NaN,7.752907,5.058995,3.647862,1.724409,0.626512
4,Mixture,1.708988,-0.186147,1.229591,1.189601,1670.951838,NaN,NaN,NaN,NaN,NaN


In [4]:
model2color = {
    "Mixture": "#ad494a",
    "PBM": "#3182bd",
    "UBM": "#6baed6",
    "DBN": "#31a354",
    "SDBN": "#74c476",
    "CM": "#fd8d3c",
    "CCM": "#fdae6b",
    "DCM": "#fdd0a2",
    "DCTR": "#969696",
    "RCTR": "#969696",
    "GCTR": "#bdbdbd",
}
df["train_time_min"] = df["train_time_s"] / 60

In [5]:
model_means = df.groupby('model')['test_ppl'].mean().sort_values()
sorted_models = model_means.index.tolist()

base = alt.Chart(df, title="Perplexity", width=250, height=220)

bars = base.mark_bar().encode(
    x=alt.X("model", title="", sort=sorted_models).axis(labelAngle=45),
    y=alt.Y("mean(test_ppl)", title="").scale(zero=False, domain=(1.2, 1.34), clamp=True),
    color=alt.Color("model", title="Models", legend=None).scale(domain=list(model2color.keys()), range=list(model2color.values())),
)

errors = base.mark_errorbar(extent="ci", thickness=4).encode(
    x=alt.X("model", title="", sort=sorted_models),
    y=alt.Y("test_ppl", title="").scale(zero=False)
)

ppl_chart = (bars + errors)
ppl_chart

alt.LayerChart(...)

In [6]:
model_means = df.groupby('model')['test_cond_ppl'].mean().sort_values()
sorted_models = model_means.index.tolist()

base = alt.Chart(df, title="Conditional Perplexity", width=250, height=220)

bars = base.mark_bar().encode(
    x=alt.X("model", title="", sort=sorted_models).axis(labelAngle=45),
    y=alt.Y("mean(test_cond_ppl)", title="").scale(zero=False, domain=(1.18, 1.34), clamp=True),
    color=alt.Color("model", title="Models", legend=None).scale(domain=list(model2color.keys()), range=list(model2color.values())),
)

errors = base.mark_errorbar(extent="ci", thickness=4).encode(
    x=alt.X("model", title="", sort=sorted_models),
    y=alt.Y("test_cond_ppl", title="").scale(zero=False)
)

cond_ppl_chart = (bars + errors)
cond_ppl_chart

alt.LayerChart(...)

In [7]:
model_means = df.groupby('model')['test_dcg@10'].mean().sort_values(ascending=False)
sorted_models = model_means.index.tolist()

base = alt.Chart(df, title="DCG@10", width=250, height=220)

bars = base.mark_bar().encode(
    x=alt.X("model", title="", sort=sorted_models).axis(labelAngle=45),
    y=alt.Y("mean(test_dcg@10)", title="").scale(zero=False, domain=(5.8, 8.2), clamp=True),
    color=alt.Color("model", title="Models", legend=None).scale(domain=list(model2color.keys()), range=list(model2color.values())),
)

errors = base.mark_errorbar(extent="ci", thickness=4).encode(
    x=alt.X("model", title="", sort=sorted_models),
    y=alt.Y("test_dcg@10", title="").scale(zero=False)
)

dcg_chart = (bars + errors)
dcg_chart

alt.LayerChart(...)

In [8]:
model_means = df.groupby('model')['train_time_min'].mean().sort_values()
sorted_models = model_means.index.tolist()

base = alt.Chart(df, title="Training Time (mins)", width=250, height=220)

bars = base.mark_bar().encode(
    x=alt.X("model", title="", sort=sorted_models).axis(labelAngle=45),
    y=alt.Y("mean(train_time_min)", title="").scale(domain=(0, 30)),
    color=alt.Color("model", title="Models", legend=None).scale(domain=list(model2color.keys()), range=list(model2color.values())),
)

errors = base.mark_errorbar(extent="ci", thickness=4).encode(
    x=alt.X("model", title="", sort=sorted_models),
    y=alt.Y("train_time_min", title="").scale(zero=False)
)

time_chart = (bars + errors)
time_chart

alt.LayerChart(...)

In [12]:
df[].max()

model                     UBM
test_loss            1.710595
test_ll             -0.186147
test_ppl             1.330105
test_cond_ppl        1.884497
train_time_s      1709.947374
test_dcg@10          8.082562
test_dcg@5           5.339433
test_dcg@3            3.92996
test_dcg@1           1.882462
test_mrr@10           0.63932
train_time_min      28.499123
dtype: object

In [9]:
chart = ppl_chart | cond_ppl_chart | dcg_chart
chart.save("4-baidu-ultr-features.svg")
chart

alt.HConcatChart(...)